In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
from accelerate import Accelerator

W0223 18:56:34.220000 13196 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.10.0+cu128
CUDA available: True
Device name: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
device = Accelerator().device
device

device(type='cuda')

In [4]:
system_prompt = """Ты — лингвист-аналитик. Твоя задача: найти общий смысл в сообщениях.
ПРАВИЛА:
1. Ответ строго в формате: [обобщённый смысл]
2. Никаких лишних слов, пояснений и вежливости.
3. Не пересказывай детали.
4. Обобщай людей до понятий (личность, знакомые, коллектив).
5. Будь максимально кратким."""

In [5]:
def format_chat_template(sample):
    return {
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': sample['instruction']},
            {'role': 'assistant', 'content': sample['response']}
        ]
    }

In [6]:
dataset_path = r'../../data/train_data/msg_clusters.csv'
data = load_dataset("csv", data_files=dataset_path, split="train")
dataset = data.map(format_chat_template, remove_columns=['instruction', 'response'])

dataset = dataset.train_test_split(test_size=0.2, shuffle=False)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/760 [00:00<?, ? examples/s]

In [7]:
model_id = r'google/gemma-3-270m-it'

In [8]:
from huggingface_hub import login
login()

In [10]:
model = AutoModelForCausalLM.from_pretrained(model_id, device_map='auto', dtype='auto')
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [12]:
from trl import SFTTrainer, SFTConfig
import torch

In [14]:
torch_dtype = model.dtype

checkpoint_dir = '../../models/checkpoints/gemma3-270m-ft'
learning_rate = 1e-5

args = SFTConfig(
    output_dir=checkpoint_dir,              # directory to save and repository id
    max_length=512,                         # max sequence length for model and packing of the dataset
    packing=False,                          # Groups multiple samples in the dataset into a single sequence
    num_train_epochs=5,                     # number of training epochs
    per_device_train_batch_size=4,          # batch size per device during training
    gradient_checkpointing=False,           # Caching is incompatible with gradient checkpointing
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_steps=1,                        # log every step
    eval_strategy="epoch",
    learning_rate=learning_rate,            # learning rate
    fp16=True if torch_dtype == torch.float16 else False,   # use float16 precision
    bf16=True if torch_dtype == torch.bfloat16 else False,  # use bfloat16 precision
    lr_scheduler_type="constant",           # use constant learning rate scheduler
    dataset_kwargs={
        "add_special_tokens": False, # Template with special tokens
        "append_concat_token": True, # Add EOS token as separator token between examples
    }
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=args,
    processing_class=tokenizer,
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Denis\_netrc.
wandb: Currently logged in as: salnikovdenis44 (salnikovdenis44-omstu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.401900,1.278322,1.226461,119893.000000,0.794524
2,1.479500,1.239546,1.184297,239786.000000,0.795662
3,1.280600,1.212059,1.168709,359679.000000,0.797322
4,1.435800,1.188630,1.106085,479572.000000,0.802171
5,1.083400,1.171476,1.065121,599465.000000,0.804233


TrainOutput(global_step=760, training_loss=1.228278669871782, metrics={'train_runtime': 2753.8958, 'train_samples_per_second': 1.104, 'train_steps_per_second': 0.276, 'total_flos': 489077288878080.0, 'train_loss': 1.228278669871782, 'epoch': 5.0})

In [16]:
save_dir = r'../../models/gemma3-270m-ft(1)'
trainer.save_model(save_dir)

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

# Plot the training loss
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

# Вывод дообученной модели

In [22]:
model_dir = r'../../models/gemma3_270m_ft_1'

ft_model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype='auto',
    device_map='auto'
)
ft_tokenizer = AutoTokenizer.from_pretrained(model_dir)

The tokenizer you are loading from '../../models/gemma3_270m_ft_1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [23]:
ft_model.config

Gemma3TextConfig {
  "_sliding_window_pattern": 6,
  "architectures": [
    "Gemma3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": null,
  "bos_token_id": 2,
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "final_logit_softcapping": null,
  "head_dim": 256,
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 640,
  "initializer_range": 0.02,
  "intermediate_size": 2048,
  "layer_types": [
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "model_type": "gemma3_text",
  "num_attention_heads": 4,
  "num_h

In [24]:
from transformers import pipeline

In [26]:
pipe = pipeline("text-generation", model=ft_model, tokenizer=ft_tokenizer)

def test(test_sample):
    prompt = pipe.tokenizer.apply_chat_template(test_sample["messages"][:2], tokenize=False, add_generation_prompt=True)
    print(prompt)
    outputs = pipe(prompt, max_new_tokens=128, disable_compile=True)
    
    
    print(f"Question:\n{test_sample['messages'][1]['content']}")
    print(f"Original Answer:\n{test_sample['messages'][2]['content']}")
    print(f"Generated Answer:\n{outputs[0]['generated_text'][len(prompt):].strip()}")
    print("-"*80)


for item in dataset['test']:
    test(item)

Device set to use cuda:0


<bos><start_of_turn>user
Ты — лингвист-аналитик. Твоя задача: найти общий смысл в сообщениях.
ПРАВИЛА:
1. Ответ строго в формате: [обобщённый смысл]
2. Никаких лишних слов, пояснений и вежливости.
3. Не пересказывай детали.
4. Обобщай людей до понятий (личность, знакомые, коллектив).
5. Будь максимально кратким.

Что объединяет эти сообщения:
1) Выбирай среднего (меня)
2) Я высчитываю средний балл
3) На 200р в среднем
4) Статистика
5) Я думаю для статистики нет необходимости выбирать точный вариант
6) Статистика просто покажет процентное соотношение 50/50 это то что нам и необходимо
7) Это статистика уже
8) Это влияет на статистику?
9) Я это Санте писал, но тут интересная статистика, можете посмотреть
10) Будем потом смотреть на статистику, не очень смешную как любимые слова маракасиков, но все таки<end_of_turn>
<start_of_turn>model

Question:
Что объединяет эти сообщения:
1) Выбирай среднего (меня)
2) Я высчитываю средний балл
3) На 200р в среднем
4) Статистика
5) Я думаю для статисти

KeyboardInterrupt: 

In [51]:
pipe = pipeline("text-generation", model=ft_model, tokenizer=ft_tokenizer)

def generate_answer(answer):
    message = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': answer}
    ]
    prompt = pipe.tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
    output = pipe(prompt, max_new_tokens=128, disable_compile=True)
    generated_answer = output[0]['generated_text'][len(prompt):]
    return generated_answer

Device set to use cuda:0


In [52]:
a = """Что объединяет эти сообщения:
1) Друзья играем в покер по примеру великого?
2) Меня научились играть в покер😎
(И я всё выиграл)
3) они точно в покер играют????
4) тогда валентина вильевна ещё спросила, играю ли я в шахматы
5) Кстати, я на днях изучала правила игры в бункер"""

generate_answer(a)

'про покер'